# Notebook 2 — Leakage-free 3-class ablation and paired statistical comparisons

**Uses your original submitted 5-second temporal–Welch PSD–10-biomarker architecture.** The `full_original` variant calls the identical `build_model()` defined in Notebook 1 (SE in temporal + PSD branches, **no post-fusion SE**). Additional variants keep their respective branch designs where included.

**Reviewer questions:** Does the full model beat temporal-only and temporal+PSD-no-SE *on unseen subjects*? Does the biomarker branch or SE recalibration contribute? 5-fold nested subject-disjoint training and validation, identical folds/hyperparameters, pooled subject-accuracy differences and paired tests.

**Prerequisites:** source `ds004504` dataset ZIP with `participants.tsv` and `derivatives/` (not the `Code (2).zip` of notebooks). GPU runtime strongly recommended. Set `DATA_MODE='local'` if your dataset has already been extracted. This notebook runs many independent training jobs; default 8 models × 5 folds. Run cells top to bottom, or reduce `SELECTED_VARIANTS` for a *labelled partial* run. **No findings are populated until you run training.**

**Reporting:** the reviewer-requested original AD–CN / FTD–CN paired comparison is separately reconstructed in Notebook 4; comparisons here are *new three-class, subject-held-out* comparisons, not the original binary results.

In [ ]:
# 1 — Dependencies, reproducibility and configuration
%pip -q install mne eeglabio kagglehub scikit-learn scipy pandas matplotlib openpyxl
import os, gc, sys, json, random, math, shutil, zipfile, hashlib, warnings, time, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, stats
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix, roc_auc_score, classification_report)
import mne
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import Callback, EarlyStopping, ModelCheckpoint, TerminateOnNaN, CSVLogger
from tensorflow.keras.optimizers.schedules import LearningRateSchedule
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)
mne.set_log_level('ERROR')

# ========= CHANGE HERE IF NEEDED =========
DATA_MODE = 'upload'      # 'upload' | 'kagglehub' | 'local'
LOCAL_DATA_PATH = '/content/ds004504'   # used only for DATA_MODE='local'
KAGGLE_DATASET = 'thngdngvn/openneuro-ds004504'  # public mirror used by original notebook
WORK_ROOT = Path('/content/EEG_AD_FTD_CN_GroupKFold')
N_FOLDS = 5
MAX_EPOCHS = 50
BATCH_SIZE = 32
MODEL_SEED = 42
OUTER_SEED = 2026
INNER_SEED = 2027
PATIENCE = 10
BOOTSTRAP_REPEATS = 1000
REBUILD_CACHE = False
RESUME_COMPLETED_FOLDS = True
# =========================================

TARGET_SFREQ = 256
WINDOW_SECONDS = 5
WINDOW_SAMPLES = TARGET_SFREQ * WINDOW_SECONDS
BOUNDARY_BUFFER_SECONDS = 0.5
TARGET_CHANNELS = ['Fp1','Fp2','F7','F3','Fz','F4','F8','T3','C3','Cz','C4','T4','T5','P3','Pz','P4','T6','O1','O2']
N_CHANNELS = len(TARGET_CHANNELS)
CLASS_NAMES = ['HC','AD','FTD']
LABEL_MAP = {'HC':0,'AD':1,'FTD':2}
N_CLASSES = 3
EXPECTED_SUBJECT_COUNTS = {'HC':29,'AD':36,'FTD':23}
EXPECTED_TOTAL_SUBJECTS = 88

PEAK_LEARNING_RATE=2e-4; MIN_LEARNING_RATE=2e-6; WARMUP_EPOCHS=3
WEIGHT_DECAY=1e-4; GRADIENT_CLIP_NORM=1.0; LABEL_SMOOTHING=0.02

DATA_CACHE = WORK_ROOT / 'cache'
SUBJECT_ARRAY_DIR = DATA_CACHE / 'subject_arrays'
BOUNDARY_METADATA_PATH = DATA_CACHE / 'segment_metadata_boundary_clean.csv'
SUBJECT_SUMMARY_PATH = DATA_CACHE / 'subject_segmentation_summary.csv'
RAW_WELCH_PATH = DATA_CACHE / 'welch_raw_log_features.npy'
RAW_BIOMARKER_PATH = DATA_CACHE / 'biomarkers_raw.npy'
FEATURE_MANIFEST_PATH = DATA_CACHE / 'feature_manifest.json'
RESULTS_ROOT = WORK_ROOT / 'results'
FOLDS_ROOT = RESULTS_ROOT / 'folds'
for d in [DATA_CACHE,SUBJECT_ARRAY_DIR,RESULTS_ROOT,FOLDS_ROOT]: d.mkdir(parents=True,exist_ok=True)

os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
random.seed(MODEL_SEED); np.random.seed(MODEL_SEED); tf.keras.utils.set_random_seed(MODEL_SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('Deterministic kernels unavailable:',str(e)[:150])
tf.keras.mixed_precision.set_global_policy('float32')
print('TensorFlow:',tf.__version__,' GPU:',tf.config.list_physical_devices('GPU'))
print('Output folder:',RESULTS_ROOT)

In [ ]:
# 2 — Locate data: upload ZIP (default), public Kaggle mirror or local directory
# Colab file upload is supported as requested, but very large dataset ZIPs may be
# more reliable using DATA_MODE='kagglehub'. No secret API keys are placed in this notebook.

def safe_extract_zip(archive_path, destination):
    destination=Path(destination).resolve(); destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            dest=(destination/info.filename).resolve()
            if not dest.is_relative_to(destination):
                raise ValueError('Unsafe ZIP path: '+info.filename)
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('ZIP contains unsupported symlink: '+info.filename)
        archive.extractall(destination)

def discover_dataset_root(root):
    root=Path(root)
    if not root.exists(): return None
    candidates=[root] if (root/'participants.tsv').is_file() else []
    candidates += [p.parent for p in root.rglob('participants.tsv')]
    for p in candidates:
        if (p/'derivatives').is_dir(): return p
    return None

if DATA_MODE == 'upload':
    from google.colab import files
    print('Upload the ds004504 DATASET ZIP (participants.tsv + derivatives/...):')
    uploaded=files.upload()
    zip_names=[n for n in uploaded.keys() if n.lower().endswith('.zip')]
    if not zip_names: raise ValueError('Please upload a dataset .zip, not the notebooks-only Code ZIP.')
    upload_root=Path('/content/openneuro_uploaded_data')
    for zip_name in zip_names:
        safe_extract_zip(Path('/content')/zip_name, upload_root)
    del uploaded
    DATA_ROOT=discover_dataset_root(upload_root)
elif DATA_MODE == 'kagglehub':
    import kagglehub
    mirror_path=Path(kagglehub.dataset_download(KAGGLE_DATASET))
    DATA_ROOT=discover_dataset_root(mirror_path)
    if DATA_ROOT is None:
        for zipped in mirror_path.rglob('*.zip'):
            extracted=Path('/content/openneuro_kaggle_unpacked')
            safe_extract_zip(zipped,extracted)
            DATA_ROOT=discover_dataset_root(extracted)
            if DATA_ROOT is not None: break
elif DATA_MODE == 'local':
    DATA_ROOT=discover_dataset_root(LOCAL_DATA_PATH)
else:
    raise ValueError('Set DATA_MODE to upload, kagglehub or local')
if DATA_ROOT is None:
    raise FileNotFoundError('Could not find participants.tsv and derivatives/. Check the dataset archive structure.')
PARTICIPANTS_PATH=DATA_ROOT/'participants.tsv'
DERIVATIVES_ROOT=DATA_ROOT/'derivatives'
print('Source dataset root:',DATA_ROOT)
print('participants.tsv SHA256:', hashlib.sha256(PARTICIPANTS_PATH.read_bytes()).hexdigest())

## 3. Preprocessing and boundary-clean 5 s cache

These loaders are adapted from the **original final multiclass notebook**: eyes-closed EEGLAB derivative `.set` recordings, 19 electrodes in the same order, resampling to 256 Hz, conversion to µV, 5 s **non-overlapping** windows, and exclusion of windows intersecting boundary annotations buffered by 0.5 s. Caches are disk-backed `.npy` arrays (not all loaded into RAM). For strict reproducibility, **do not change these choices halfway through the comparison**.

In [ ]:
GROUP_MAP = {
    "A": "AD", "AD": "AD", "ALZHEIMER": "AD", "ALZHEIMERS": "AD",
    "C": "HC", "CN": "HC", "HC": "HC", "CONTROL": "HC",
    "HEALTHY": "HC", "COGNITIVELY NORMAL": "HC",
    "F": "FTD", "FTD": "FTD", "FRONTOTEMPORAL DEMENTIA": "FTD",
}

CHANNEL_ALIASES = {
    "FP1": "Fp1", "FP2": "Fp2", "F7": "F7", "F3": "F3",
    "FZ": "Fz", "F4": "F4", "F8": "F8",
    "T3": "T3", "T7": "T3", "C3": "C3", "CZ": "Cz",
    "C4": "C4", "T4": "T4", "T8": "T4",
    "T5": "T5", "P7": "T5", "P3": "P3", "PZ": "Pz",
    "P4": "P4", "T6": "T6", "P8": "T6", "O1": "O1", "O2": "O2",
}

def normalize_group(value):
    text = str(value).strip().upper().replace("_", " ").replace("-", " ")
    text = " ".join(text.split())
    if text not in GROUP_MAP:
        raise ValueError(f"Unknown group value: {value!r}")
    return GROUP_MAP[text]


def load_participants():
    participants = pd.read_csv(
        PARTICIPANTS_PATH,
        sep="\t",
        na_values=["n/a", "N/A", "NA", "nan", ""],
    )
    participants["participant_id"] = participants["participant_id"].astype(str).str.strip()

    group_column = next(
        (c for c in participants.columns if c.lower() in {"group", "diagnosis", "label"}),
        None,
    )
    if group_column is None:
        raise RuntimeError("No group/diagnosis column found.")

    participants["group"] = participants[group_column].apply(normalize_group)
    participants["label"] = participants["group"].map(LABEL_MAP).astype(np.int32)
    participants = participants[["participant_id", "group", "label"]].copy()

    observed = participants["group"].value_counts().to_dict()
    for group_name, expected in EXPECTED_SUBJECT_COUNTS.items():
        assert int(observed.get(group_name, 0)) == expected
    assert len(participants) == EXPECTED_TOTAL_SUBJECTS
    assert participants["participant_id"].is_unique
    return participants

def canonicalize_channels(raw):
    renames = {}
    for channel in raw.ch_names:
        key = str(channel).strip().upper().replace(" ", "")
        canonical = CHANNEL_ALIASES.get(key, channel)
        if canonical != channel:
            renames[channel] = canonical
    if renames:
        raw.rename_channels(renames)
    return raw

def find_derivative_file(participant_id):
    eeg_dir = DERIVATIVES_ROOT / participant_id / "eeg"
    preferred = eeg_dir / f"{participant_id}_task-eyesclosed_eeg.set"
    if preferred.exists():
        return preferred
    matches = sorted(eeg_dir.glob("*task-eyesclosed*_eeg.set"))
    if matches:
        return matches[0] # Fixed Bug 1: Explicitly return file path element
    raise FileNotFoundError(f"No derivative EEG file for {participant_id}")

def get_boundary_intervals(raw, buffer_seconds=0.5):
    duration = raw.n_times / float(raw.info["sfreq"])
    intervals = []
    for onset, annotation_duration, description in zip(
        raw.annotations.onset,
        raw.annotations.duration,
        raw.annotations.description,
    ):
        if "boundary" not in str(description).lower():
            continue
        relative_onset = float(onset) - float(raw.first_time)
        if relative_onset < -1.0 or relative_onset > duration + 1.0:
            relative_onset = float(onset)
        start = max(0.0, relative_onset - buffer_seconds)
        end = min(
            duration,
            relative_onset + max(0.0, float(annotation_duration)) + buffer_seconds,
        )
        if end <= start:
            end = min(duration, start + 2.0 * buffer_seconds)
        intervals.append((start, end))
    return sorted(intervals)

def segment_overlaps_boundary(start_seconds, end_seconds, intervals):
    return any(
        start_seconds < boundary_end and end_seconds > boundary_start
        for boundary_start, boundary_end in intervals
    )

def load_and_prepare_subject(participant_id):
    file_path = find_derivative_file(participant_id)
    raw = mne.io.read_raw_eeglab(
        str(file_path),
        preload=True,
        verbose=False,
    )
    original_sfreq = float(raw.info["sfreq"])
    boundary_intervals = get_boundary_intervals(
        raw,
        buffer_seconds=BOUNDARY_BUFFER_SECONDS,
    )
    raw = canonicalize_channels(raw)
    missing = [ch for ch in TARGET_CHANNELS if ch not in raw.ch_names]
    if missing:
        raise RuntimeError(f"{participant_id}: missing channels {missing}")
    raw.pick(TARGET_CHANNELS)
    raw.reorder_channels(TARGET_CHANNELS)
    if int(round(raw.info["sfreq"])) != TARGET_SFREQ:
        raw.resample(TARGET_SFREQ, npad="auto", verbose=False)
    data_uv = (raw.get_data().T * 1e6).astype(np.float32)

    # Fixed Bug 2: Evaluate 2D index shape column count correctly
    if data_uv.shape[1] != N_CHANNELS or not np.isfinite(data_uv).all():
        raise RuntimeError(f"{participant_id}: invalid prepared EEG array")
    return data_uv, boundary_intervals, file_path, original_sfreq

def calculate_segment_quality(segment_uv):
    channel_ptp = np.ptp(segment_uv, axis=0)
    channel_std = np.std(segment_uv, axis=0)
    return {
        "max_abs_uv": float(np.max(np.abs(segment_uv))),
        "max_ptp_uv": float(np.max(channel_ptp)),
        "median_ptp_uv": float(np.median(channel_ptp)),
        "min_channel_std_uv": float(np.min(channel_std)),
        "median_channel_std_uv": float(np.median(channel_std)),
    }

In [ ]:
# 4 — Create or load cleaned per-subject segments
if REBUILD_CACHE:
    for f in [BOUNDARY_METADATA_PATH, SUBJECT_SUMMARY_PATH,RAW_WELCH_PATH,RAW_BIOMARKER_PATH,FEATURE_MANIFEST_PATH]:
        if f.exists(): f.unlink()
    for f in SUBJECT_ARRAY_DIR.glob('*_5s_boundary_clean.npy'): f.unlink()

participants_df=load_participants()
part_hash=hashlib.sha256(PARTICIPANTS_PATH.read_bytes()).hexdigest()
cache_ok=BOUNDARY_METADATA_PATH.exists() and SUBJECT_SUMMARY_PATH.exists()
if cache_ok:
    boundary_metadata_df=pd.read_csv(BOUNDARY_METADATA_PATH)
    subject_summary_df=pd.read_csv(SUBJECT_SUMMARY_PATH)
    for p in boundary_metadata_df['subject_array_path'].drop_duplicates():
        if not Path(p).exists(): cache_ok=False; break
    if set(boundary_metadata_df['participant_id']) != set(participants_df['participant_id']):
        cache_ok=False
if not cache_ok:
    rows=[]; subject_rows=[]
    for i, row in participants_df.reset_index(drop=True).iterrows():
        sid, group, label=row['participant_id'], row['group'], int(row['label'])
        data, boundaries, file_path, original_sfreq=load_and_prepare_subject(sid)
        total=data.shape[0]//WINDOW_SAMPLES
        selected=[]; rejected=0
        for seg in range(total):
            start=seg*WINDOW_SAMPLES; end=start+WINDOW_SAMPLES
            ts,te=start/TARGET_SFREQ,end/TARGET_SFREQ
            if segment_overlaps_boundary(ts,te,boundaries): rejected+=1; continue
            segment=data[start:end]
            if segment.shape != (WINDOW_SAMPLES,N_CHANNELS): continue
            selected.append(segment)
            rows.append({'participant_id':sid,'group':group,'label':label,
                         'local_segment_index':len(selected)-1,'original_segment_index':seg,
                         'start_sec':ts,'end_sec':te,
                         'subject_array_path':str(SUBJECT_ARRAY_DIR/f'{sid}_5s_boundary_clean.npy')})
        if not selected:
            raise RuntimeError(f'No valid 5-second segments for {sid}; do not silently drop subject.')
        np.save(SUBJECT_ARRAY_DIR/f'{sid}_5s_boundary_clean.npy',np.stack(selected).astype(np.float32))
        subject_rows.append({'participant_id':sid,'group':group,'label':label,
          'source_file':str(file_path),'original_sfreq':original_sfreq,
          'recording_duration_sec':data.shape[0]/TARGET_SFREQ,
          'boundary_annotations':len(boundaries),'total_complete_windows':total,
          'rejected_boundary_windows':rejected,'clean_windows':len(selected)})
        print(f'[{i+1}/{len(participants_df)}] {sid} {group}: {len(selected)} valid windows')
        del data,selected
        gc.collect()
    boundary_metadata_df=pd.DataFrame(rows).reset_index(drop=True)
    subject_summary_df=pd.DataFrame(subject_rows)
    boundary_metadata_df.to_csv(BOUNDARY_METADATA_PATH,index=False)
    subject_summary_df.to_csv(SUBJECT_SUMMARY_PATH,index=False)

boundary_metadata_df=boundary_metadata_df.sort_values(['participant_id','local_segment_index']).reset_index(drop=True)
boundary_metadata_df['segment_uid']=boundary_metadata_df['participant_id'].astype(str)+'__clean_'+boundary_metadata_df['local_segment_index'].astype(str).str.zfill(4)
boundary_metadata_df['feature_index']=np.arange(len(boundary_metadata_df))
assert boundary_metadata_df['segment_uid'].is_unique
assert boundary_metadata_df['participant_id'].nunique()==88
assert (boundary_metadata_df.groupby('participant_id')['label'].nunique()==1).all()
assert boundary_metadata_df.groupby('group')['participant_id'].nunique().to_dict()==EXPECTED_SUBJECT_COUNTS
assert boundary_metadata_df['subject_array_path'].map(lambda p:Path(p).exists()).all()
metadata_sha256=hashlib.sha256(boundary_metadata_df[['segment_uid','label','start_sec','end_sec']].to_csv(index=False).encode()).hexdigest()
print('Subjects and segments:',boundary_metadata_df['participant_id'].nunique(),len(boundary_metadata_df))
print('Per class:',boundary_metadata_df.groupby('group').agg(subjects=('participant_id','nunique'),segments=('segment_uid','size')))
if len(boundary_metadata_df)!=13293: print('NOTE: cleaned segment count differs from the original manuscript (13293); investigate dataset/derivative differences.')
subject_summary_df.to_csv(RESULTS_ROOT/'subject_recording_duration_and_segments.csv',index=False)

## 5. Welch PSD maps and 10 biomarkers

Exactly the three-input design from your original multiclass experiment: raw µV EEG `(1280, 19, 1)`, log absolute + log relative Welch PSD `(89, 19, 2)`, and the 10-dimensional spectral descriptor vector. This cell caches **unnormalized** spectral features; **fold-specific training-only mean/std** is fitted later. This is essential: the original global standardized feature cache must **not** be used for GroupKFold.

In [ ]:
# 5 — Feature parameters (matching original multiclass notebook)
WELCH_WINDOW_SAMPLES=512; WELCH_OVERLAP_SAMPLES=256; WELCH_NFFT=512
WELCH_FMIN=1.0; WELCH_FMAX=45.0; WELCH_N_FREQUENCIES=89
N_BIOMARKERS=10; FEATURE_BUILD_BATCH_SIZE=32; NORMALIZATION_BATCH_SIZE=256
WELCH_CLIP_VALUE=5.0; BIOMARKER_CLIP_VALUE=5.0; PSD_EPSILON=1e-12; STD_EPSILON=1e-6
POSTERIOR_INDICES=np.array([TARGET_CHANNELS.index(s) for s in ['T5','P3','Pz','P4','T6','O1','O2']])
ANTERIOR_INDICES=np.array([TARGET_CHANNELS.index(s) for s in ['Fp1','Fp2','F7','F3','Fz','F4','F8']])
OCCIPITAL_INDICES=np.array([TARGET_CHANNELS.index(s) for s in ['O1','O2']])
TEMPORAL_INDICES=np.array([TARGET_CHANNELS.index(s) for s in ['T3','T4','T5','T6']])
BIOMARKER_NAMES=['posterior_relative_alpha','log_posterior_anterior_alpha_ratio',
 'posterior_alpha_peak_hz','log_global_theta_alpha_ratio','log_global_slowing_ratio',
 'frontal_relative_theta','occipital_relative_alpha','temporal_relative_alpha',
 'log_global_total_power','log_posterior_total_power']

In [ ]:
def integrate_band(psd, frequencies, low, high, include_high=False):
    mask = (
        (frequencies >= low) & (frequencies <= high)
        if include_high
        else (frequencies >= low) & (frequencies < high)
    )
    if mask.sum() < 2:
        raise RuntimeError(f"Insufficient bins for {low}-{high} Hz")
    if hasattr(np, "trapezoid"):
        return np.trapezoid(psd[:, mask, :], frequencies[mask], axis=1)
    return np.trapz(psd[:, mask, :], frequencies[mask], axis=1)


def compute_features(segments_uv):
    segments_uv = np.asarray(segments_uv, dtype=np.float32)
    assert segments_uv.shape[1:] == (WINDOW_SAMPLES, N_CHANNELS)

    frequencies, psd = signal.welch(
        segments_uv,
        fs=TARGET_SFREQ,
        window="hann",
        nperseg=WELCH_WINDOW_SAMPLES,
        noverlap=WELCH_OVERLAP_SAMPLES,
        nfft=WELCH_NFFT,
        detrend="constant",
        return_onesided=True,
        scaling="density",
        axis=1,
        average="mean",
    )

    mask = (frequencies >= WELCH_FMIN) & (frequencies <= WELCH_FMAX)
    frequencies = frequencies[mask].astype(np.float32)
    psd = psd[:, mask, :].astype(np.float64)

    assert len(frequencies) == WELCH_N_FREQUENCIES

    log_abs = np.log(psd + PSD_EPSILON)
    rel_psd = psd / (np.sum(psd, axis=1, keepdims=True) + PSD_EPSILON)
    log_rel = np.log(rel_psd + PSD_EPSILON)
    welch_maps = np.stack([log_abs, log_rel], axis=-1).astype(np.float32)

    delta = integrate_band(psd, frequencies, 1.0, 4.0)
    theta = integrate_band(psd, frequencies, 4.0, 8.0)
    alpha = integrate_band(psd, frequencies, 8.0, 13.0)
    beta = integrate_band(psd, frequencies, 13.0, 30.0)
    total = integrate_band(psd, frequencies, 1.0, 45.0, include_high=True)

    rel_theta = theta / (total + PSD_EPSILON)
    rel_alpha = alpha / (total + PSD_EPSILON)

    posterior_alpha = np.mean(alpha[:, POSTERIOR_INDICES], axis=1)
    anterior_alpha = np.mean(alpha[:, ANTERIOR_INDICES], axis=1)

    posterior_psd = np.mean(psd[:, :, POSTERIOR_INDICES], axis=2)
    peak_mask = (frequencies >= 7.0) & (frequencies <= 13.0)
    peak_frequencies = frequencies[peak_mask]
    peak_hz = peak_frequencies[np.argmax(posterior_psd[:, peak_mask], axis=1)]

    global_delta = np.mean(delta, axis=1)
    global_theta = np.mean(theta, axis=1)
    global_alpha = np.mean(alpha, axis=1)
    global_beta = np.mean(beta, axis=1)
    global_total = np.mean(total, axis=1)
    posterior_total = np.mean(total[:, POSTERIOR_INDICES], axis=1)

    biomarkers = np.column_stack(
        [
            np.mean(rel_alpha[:, POSTERIOR_INDICES], axis=1),
            np.log((posterior_alpha + PSD_EPSILON) / (anterior_alpha + PSD_EPSILON)),
            peak_hz,
            np.log((global_theta + PSD_EPSILON) / (global_alpha + PSD_EPSILON)),
            np.log(
                (global_delta + global_theta + PSD_EPSILON)
                / (global_alpha + global_beta + PSD_EPSILON)
            ),
            np.mean(rel_theta[:, ANTERIOR_INDICES], axis=1),
            np.mean(rel_alpha[:, OCCIPITAL_INDICES], axis=1),
            np.mean(rel_alpha[:, TEMPORAL_INDICES], axis=1),
            np.log(global_total + PSD_EPSILON),
            np.log(posterior_total + PSD_EPSILON),
        ]
    ).astype(np.float32)

    assert np.isfinite(welch_maps).all()
    assert np.isfinite(biomarkers).all()
    return welch_maps, biomarkers, frequencies


def training_stats(array, indices, batch_size=256):
    shape = array.shape[1:]
    total = np.zeros(shape, dtype=np.float64)
    total_sq = np.zeros(shape, dtype=np.float64)
    count = 0

    for start in range(0, len(indices), batch_size):
        batch = np.asarray(array[indices[start:start + batch_size]], dtype=np.float64)
        total += np.sum(batch, axis=0)
        total_sq += np.sum(np.square(batch), axis=0)
        count += len(batch)

    mean = total / count
    variance = np.maximum(total_sq / count - np.square(mean), STD_EPSILON ** 2)
    return mean.astype(np.float32), np.sqrt(variance).astype(np.float32)

In [ ]:
# 6 — Build the RAW unsupervised spectral caches (no global standardization)
cache_fingerprint={'participants_sha256':part_hash,'metadata_sha256':metadata_sha256,
                   'segments':len(boundary_metadata_df),'channels':TARGET_CHANNELS,
                   'sampling_hz':TARGET_SFREQ,'window_seconds':WINDOW_SECONDS,
                   'welch_nperseg':WELCH_WINDOW_SAMPLES,'welch_noverlap':WELCH_OVERLAP_SAMPLES}
feature_ok=all(p.exists() for p in [RAW_WELCH_PATH,RAW_BIOMARKER_PATH,FEATURE_MANIFEST_PATH])
if feature_ok:
    try:
        assert json.loads(FEATURE_MANIFEST_PATH.read_text())==cache_fingerprint
        assert np.load(RAW_WELCH_PATH,mmap_mode='r').shape==(len(boundary_metadata_df),89,19,2)
        assert np.load(RAW_BIOMARKER_PATH,mmap_mode='r').shape==(len(boundary_metadata_df),10)
    except Exception: feature_ok=False
if not feature_ok:
    n=len(boundary_metadata_df)
    fw=np.lib.format.open_memmap(RAW_WELCH_PATH,mode='w+',dtype=np.float32,shape=(n,89,19,2))
    fb=np.lib.format.open_memmap(RAW_BIOMARKER_PATH,mode='w+',dtype=np.float32,shape=(n,10))
    for count,(path,sub) in enumerate(boundary_metadata_df.groupby('subject_array_path',sort=False),start=1):
        arr=np.load(path,mmap_mode='r')
        positions=sub.index.to_numpy(dtype=np.int64)
        local=sub['local_segment_index'].to_numpy(dtype=np.int64)
        for start in range(0,len(local),FEATURE_BUILD_BATCH_SIZE):
            stop=min(start+FEATURE_BUILD_BATCH_SIZE,len(local))
            data=np.asarray(arr[local[start:stop]],dtype=np.float32)
            w,b,_=compute_features(data)
            fw[positions[start:stop]]=w
            fb[positions[start:stop]]=b
        print(f'Features: {count:02d}/88 — {sub.participant_id.iloc[0]}')
    fw.flush(); fb.flush()
    del fw,fb
    FEATURE_MANIFEST_PATH.write_text(json.dumps(cache_fingerprint,indent=2))
welch_raw=np.load(RAW_WELCH_PATH,mmap_mode='r')
biomarkers_raw=np.load(RAW_BIOMARKER_PATH,mmap_mode='r')
assert welch_raw.shape[0]==biomarkers_raw.shape[0]==len(boundary_metadata_df)
print('Welch:',welch_raw.shape,'Biomarkers:',biomarkers_raw.shape)

## 6. Independent-subject splits and original 3-input architecture

Outer 5-fold StratifiedGroupKFold uses **one record per unique participant** to balance 36 AD, 23 FTD and 29 HC subjects. Within each outer training group, a separate subject-grouped split selects validation subjects for early stopping. Test labels are used **only after** validation checkpoint selection. Fixed original hyperparameters are used (no fold-specific tuning against test performance).

The architecture below was copied from your final multiclass notebook, including the **absence** of a post-fusion SE block in the code (there are SE blocks **inside the temporal and frequency branches**). If you later add post-fusion SE, that becomes a **different architecture** requiring a fresh comparison.

In [ ]:
# 7 — Create all 5 subject-level train/validation/test folds; save a full split audit
subj=participants_df[['participant_id','group','label']].sort_values('participant_id').reset_index(drop=True)
outer=StratifiedGroupKFold(n_splits=N_FOLDS,shuffle=True,random_state=OUTER_SEED)
fold_splits=[]; audit_rows=[]
for fold,(dev_idx,test_idx) in enumerate(outer.split(subj,subj.label,groups=subj.participant_id),start=1):
    dev=subj.iloc[dev_idx].reset_index(drop=True)
    test=subj.iloc[test_idx].reset_index(drop=True)
    inner=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=INNER_SEED+fold)
    tr_idx,val_idx=next(inner.split(dev,dev.label,groups=dev.participant_id))
    train=dev.iloc[tr_idx]; val=dev.iloc[val_idx]
    sets={'train':set(train.participant_id),'validation':set(val.participant_id),'test':set(test.participant_id)}
    assert sets['train'].isdisjoint(sets['validation'])
    assert sets['train'].isdisjoint(sets['test'])
    assert sets['validation'].isdisjoint(sets['test'])
    assert len(sets['train']|sets['validation']|sets['test'])==88
    for part,table in [('train',train),('validation',val),('test',test)]:
        assert set(table.label)=={0,1,2}, f'Fold {fold}: {part} missing diagnostic group'
        for _,r in table.iterrows():
            audit_rows.append({'fold':fold,'partition':part,'participant_id':r.participant_id,
                               'group':r.group,'label':int(r.label)})
    fold_splits.append((fold,sets))

split_audit=pd.DataFrame(audit_rows)
assert (split_audit.groupby(['fold','participant_id']).size()==1).all()
assert (split_audit[split_audit.partition=='test'].groupby('participant_id').size()==1).all()
assert split_audit.query("partition=='test'").participant_id.nunique()==88
split_audit.to_csv(RESULTS_ROOT/'subject_split_manifest.csv',index=False)
print('Per fold distinct subjects:')
display(split_audit.groupby(['fold','partition','group']).size().unstack(fill_value=0))
print('PASS: 88 test subjects exactly once; zero overlapping participant IDs in each fold.')

In [ ]:
if not hasattr(layers, "GroupNormalization"):
    raise RuntimeError("Current Keras GroupNormalization is required.")

@tf.keras.utils.register_keras_serializable(package="EEG")
class ChannelSegmentZScoreClip(layers.Layer):
    def __init__(self, clip_value=6.0, epsilon=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.clip_value = float(clip_value)
        self.epsilon = float(epsilon)

    def call(self, inputs):
        mean = tf.reduce_mean(inputs, axis=1, keepdims=True)
        variance = tf.reduce_mean(tf.square(inputs - mean), axis=1, keepdims=True)
        normalized = (inputs - mean) / tf.sqrt(variance + self.epsilon)
        return tf.clip_by_value(normalized, -self.clip_value, self.clip_value)

    def get_config(self):
        config = super().get_config()
        config.update({"clip_value": self.clip_value, "epsilon": self.epsilon})
        return config


@tf.keras.utils.register_keras_serializable(package="EEG")
class SparseLabelSmoothedCrossentropy(tf.keras.losses.Loss):
    def __init__(self, num_classes=3, label_smoothing=0.02, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = int(num_classes)
        self.label_smoothing = float(label_smoothing)

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        one_hot = tf.one_hot(y_true, depth=self.num_classes)
        return tf.keras.losses.categorical_crossentropy(
            one_hot,
            y_pred,
            label_smoothing=self.label_smoothing,
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {"num_classes": self.num_classes, "label_smoothing": self.label_smoothing}
        )
        return config


@tf.keras.utils.register_keras_serializable(package="EEG")
class WarmupCosineDecay(LearningRateSchedule):
    def __init__(self, peak_lr, min_lr, warmup_steps, total_steps, **kwargs):
        super().__init__()
        self.peak_lr = float(peak_lr)
        self.min_lr = float(min_lr)
        self.warmup_steps = int(warmup_steps)
        self.total_steps = int(total_steps)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(max(self.warmup_steps, 1), tf.float32)
        total_steps = tf.cast(max(self.total_steps, self.warmup_steps + 1), tf.float32)
        warmup_lr = self.peak_lr * step / warmup_steps
        progress = tf.clip_by_value(
            (step - warmup_steps) / (total_steps - warmup_steps),
            0.0,
            1.0,
        )
        cosine_lr = self.min_lr + (self.peak_lr - self.min_lr) * 0.5 * (
            1.0 + tf.cos(tf.constant(np.pi, tf.float32) * progress)
        )
        return tf.where(step < warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            "peak_lr": self.peak_lr,
            "min_lr": self.min_lr,
            "warmup_steps": self.warmup_steps,
            "total_steps": self.total_steps,
        }


def se_block(x, ratio, prefix):
    channels = int(x.shape[-1])
    a = layers.GlobalAveragePooling2D(name=f"{prefix}_gap")(x)
    a = layers.Dense(max(channels // ratio, 8), activation="gelu", name=f"{prefix}_fc1")(a)
    a = layers.Dense(channels, activation="sigmoid", name=f"{prefix}_fc2")(a)
    a = layers.Reshape((1, 1, channels), name=f"{prefix}_reshape")(a)
    return layers.Multiply(name=f"{prefix}_scale")([x, a])


def cgn(x, filters, kernel, groups, name, separable=False):
    conv = layers.SeparableConv2D if separable else layers.Conv2D
    x = conv(
        filters,
        kernel,
        padding="same",
        use_bias=False,
        name=f"{name}_conv",
    )(x)
    x = layers.GroupNormalization(
        groups=groups,
        axis=-1,
        epsilon=1e-5,
        name=f"{name}_gn",
    )(x)
    return layers.Activation("elu", name=f"{name}_elu")(x)


def build_model():
    raw_input = layers.Input((1280, 19, 1), name="raw_eeg_microvolts")
    welch_input = layers.Input((89, 19, 2), name="welch_absolute_relative_psd")
    biomarker_input = layers.Input((10,), name="spectral_biomarkers")

    # Temporal branch
    t0 = ChannelSegmentZScoreClip(name="temporal_zscore")(raw_input)
    t1 = cgn(t0, 32, (32, 1), 8, "time32")
    t2 = cgn(t0, 32, (64, 1), 8, "time64")
    t3 = cgn(t0, 32, (128, 1), 8, "time128")
    t = layers.Concatenate(name="time_concat")([t1, t2, t3])
    t = layers.DepthwiseConv2D((1, 19), depth_multiplier=2, use_bias=False, name="time_spatial")(t)
    t = layers.GroupNormalization(groups=16, name="time_spatial_gn")(t)
    t = layers.Activation("elu")(t)
    t = se_block(t, 8, "time_se1")
    t = layers.AveragePooling2D((4, 1))(t)
    t = layers.SpatialDropout2D(0.15)(t)
    t = cgn(t, 192, (16, 1), 16, "time_sep1", separable=True)
    t = layers.AveragePooling2D((4, 1))(t)
    t = layers.SpatialDropout2D(0.20)(t)
    t = cgn(t, 256, (8, 1), 16, "time_sep2", separable=True)
    t = se_block(t, 8, "time_se2")
    t = layers.AveragePooling2D((2, 1))(t)
    t = layers.SpatialDropout2D(0.25)(t)
    t = layers.GlobalAveragePooling2D(name="time_embedding")(t)

    # Welch branch
    f = cgn(welch_input, 32, (7, 1), 8, "freq1")
    f = cgn(f, 48, (5, 1), 8, "freq2")
    f = layers.DepthwiseConv2D((1, 19), depth_multiplier=2, use_bias=False, name="freq_spatial")(f)
    f = layers.GroupNormalization(groups=16, name="freq_spatial_gn")(f)
    f = layers.Activation("elu")(f)
    f = se_block(f, 8, "freq_se1")
    f = layers.AveragePooling2D((2, 1))(f)
    f = layers.SpatialDropout2D(0.15)(f)
    f = cgn(f, 128, (5, 1), 16, "freq_sep1", separable=True)
    f = layers.AveragePooling2D((2, 1))(f)
    f = layers.SpatialDropout2D(0.20)(f)
    f = cgn(f, 192, (3, 1), 16, "freq_sep2", separable=True)
    f = se_block(f, 8, "freq_se2")
    f = layers.SpatialDropout2D(0.25)(f)
    f = layers.GlobalAveragePooling2D(name="freq_embedding")(f)

    # Biomarkers
    b = layers.Dense(32, activation="gelu", name="bio_dense1")(biomarker_input)
    b = layers.Dropout(0.15)(b)
    b = layers.Dense(32, activation="gelu", name="bio_embedding")(b)

    # Fusion
    z = layers.Concatenate(name="fusion")([t, f, b])
    z = layers.Dense(256, activation="gelu", name="fusion_dense1")(z)
    z = layers.LayerNormalization(name="fusion_ln1")(z)
    z = layers.Dropout(0.30)(z)
    z = layers.Dense(96, activation="gelu", name="fusion_dense2")(z)
    z = layers.LayerNormalization(name="fusion_ln2")(z)
    z = layers.Dropout(0.20)(z)
    output = layers.Dense(3, activation="softmax", name="multiclass_output")(z)

    return Model(
        inputs={
            "raw_eeg_microvolts": raw_input,
            "welch_absolute_relative_psd": welch_input,
            "spectral_biomarkers": biomarker_input,
        },
        outputs=output,
    )

In [ ]:
# 8 — Sanity-check exact model architecture (from the manuscript's multiclass notebook)
model_check=build_model()
print('Trainable parameter count:',model_check.count_params())
if model_check.count_params()!=339743:
    print('WARNING: parameter count differs from 339,743 reported in your paper. Check TensorFlow/Keras and the original implementation.')
assert {i.name.split(':')[0].split('/')[-1] for i in model_check.inputs}=={
 'raw_eeg_microvolts','welch_absolute_relative_psd','spectral_biomarkers'}
assert model_check.output_shape[-1]==3
del model_check
tf.keras.backend.clear_session()

## 7. Fold-specific generators, training, and evaluation

- Spectral normalization statistics are fitted **only** on training-subject segments (not validation/test).
- For model checkpointing, validation macro-F1 is calculated at the **segment** level, as in your original model. Evaluation reports **both segment metrics and subject-aggregated metrics**; the latter is the primary result for the reviewers.
- The checkpoint is selected without inspecting test labels. Each test subject is predicted in exactly one fold. Save `.weights.h5`, normalizer stats and per-fold predictions for reproducibility.

In [ ]:
# 9 — Dataset sequence and helpers
BaseDataset=getattr(tf.keras.utils,'PyDataset',tf.keras.utils.Sequence)
class MultiInputSequence(BaseDataset):
    def __init__(self, frame, stats_dict, batch_size=BATCH_SIZE, shuffle=False, seed=42):
        try: super().__init__()
        except TypeError: super().__init__()
        self.frame=frame.reset_index(drop=True)
        self.bs=batch_size; self.shuffle=shuffle; self.rng=np.random.default_rng(seed)
        self.order=np.arange(len(self.frame),dtype=np.int64)
        self.welch=np.load(RAW_WELCH_PATH,mmap_mode='r')
        self.biom=np.load(RAW_BIOMARKER_PATH,mmap_mode='r')
        self.stats=stats_dict
        self.paths=self.frame.subject_array_path.astype(str).to_numpy()
        self.locals=self.frame.local_segment_index.to_numpy(np.int64)
        self.features=self.frame.feature_index.to_numpy(np.int64)
        self.y=self.frame.label.to_numpy(np.int32)
        self.mm={}
        self.on_epoch_end()
    def __len__(self): return int(math.ceil(len(self.frame)/self.bs))
    def on_epoch_end(self):
        if self.shuffle: self.rng.shuffle(self.order)
    def __getitem__(self,i):
        ids=self.order[i*self.bs:(i+1)*self.bs]
        raw=np.empty((len(ids),WINDOW_SAMPLES,N_CHANNELS,1),dtype=np.float32)
        for j,r in enumerate(ids):
            path=self.paths[r]
            if path not in self.mm: self.mm[path]=np.load(path,mmap_mode='r')
            raw[j,:,:,0]=self.mm[path][self.locals[r]]
        fi=self.features[ids]
        w=(np.asarray(self.welch[fi],dtype=np.float32)-self.stats['welch_mean'])/self.stats['welch_std']
        b=(np.asarray(self.biom[fi],dtype=np.float32)-self.stats['bio_mean'])/self.stats['bio_std']
        w=np.clip(w,-WELCH_CLIP_VALUE,WELCH_CLIP_VALUE)
        b=np.clip(b,-BIOMARKER_CLIP_VALUE,BIOMARKER_CLIP_VALUE)
        assert np.isfinite(w).all() and np.isfinite(b).all() and np.isfinite(raw).all()
        return ({'raw_eeg_microvolts':raw,
                 'welch_absolute_relative_psd':w,'spectral_biomarkers':b},self.y[ids])

def frame_for_subjects(subject_ids):
    x=boundary_metadata_df[boundary_metadata_df.participant_id.isin(subject_ids)].copy().reset_index(drop=True)
    assert set(x.participant_id)==set(subject_ids)
    return x

def fold_normalizers(train_frame):
    idx=train_frame.feature_index.to_numpy(dtype=np.int64)
    wm,ws=training_stats(welch_raw,idx,NORMALIZATION_BATCH_SIZE)
    bm,bs=training_stats(biomarkers_raw,idx,NORMALIZATION_BATCH_SIZE)
    assert np.isfinite(wm).all() and np.isfinite(ws).all() and (ws>0).all()
    return {'welch_mean':wm,'welch_std':ws,'bio_mean':bm,'bio_std':bs}

def multiclass_metrics(y,p):
    y=np.asarray(y,dtype=np.int32);p=np.asarray(p,dtype=np.float64)
    pred=np.argmax(p,axis=1)
    metrics={'accuracy':accuracy_score(y,pred),
             'balanced_accuracy':balanced_accuracy_score(y,pred),
             'macro_f1':f1_score(y,pred,labels=[0,1,2],average='macro',zero_division=0),
             'weighted_f1':f1_score(y,pred,labels=[0,1,2],average='weighted',zero_division=0),
             'macro_precision':precision_score(y,pred,labels=[0,1,2],average='macro',zero_division=0),
             'macro_recall':recall_score(y,pred,labels=[0,1,2],average='macro',zero_division=0)}
    metrics['macro_auc_ovr']=roc_auc_score(y,p,labels=[0,1,2],multi_class='ovr',average='macro') if len(np.unique(y))==3 else np.nan
    return metrics

def aggregate_subject_predictions(segment_predictions):
    # Each participant has ONE true diagnosis and ONE outer test fold.
    assert (segment_predictions.groupby('participant_id').label.nunique()==1).all()
    assert (segment_predictions.groupby('participant_id').fold.nunique()==1).all()
    subj_pred=segment_predictions.groupby(['fold','participant_id','group','label'],as_index=False).agg(
        n_segments=('segment_uid','size'),
        segment_accuracy=('segment_correct','mean'),
        p_HC=('p_HC','mean'),p_AD=('p_AD','mean'),p_FTD=('p_FTD','mean'))
    probs=subj_pred[['p_HC','p_AD','p_FTD']].to_numpy()
    subj_pred['subject_prediction']=probs.argmax(axis=1)
    subj_pred['subject_correct']=(subj_pred.subject_prediction==subj_pred.label).astype(int)
    return subj_pred

class ValidationMacroF1(Callback):
    def __init__(self,valid_sequence):
        super().__init__();self.valid_sequence=valid_sequence
    def on_epoch_end(self,epoch,logs=None):
        logs=logs if logs is not None else {}
        pred=self.model.predict(self.valid_sequence,verbose=0)
        label=self.valid_sequence.y
        logs['val_macro_f1']=float(f1_score(label,pred.argmax(axis=1),labels=[0,1,2],average='macro',zero_division=0))
        logs['val_balanced_accuracy']=float(balanced_accuracy_score(label,pred.argmax(axis=1)))
        print(f' — val_macro_f1={logs["val_macro_f1"]:.4f} (subject-disjoint validation)')

## New: ablation variants (original full architecture + controlled reductions)

All variants share **exact same subject-grouped folds**, segmentation, training-subject normalization, validation selection, and optimization. Branch SE is toggled using a controlled branch-level change. `full_original` is *not modified*. Missing inputs are dropped rather than replaced with zeros. Param counts are saved per variant. **Multiple comparisons are corrected by Holm's procedure**; fold-level tests are exploratory with only five folds.

In [ ]:
# The ORIGINAL full model remains `build_model()` from Notebook 1.
VARIANT_BRANCHES = {
    'full_original':('time','freq','bio'),
    'full_no_se':('time','freq','bio'),
    'time_only':('time',),
    'time_only_no_se':('time',),
    'psd_only':('freq',),
    'biomarker_only':('bio',),
    'time_psd_se':('time','freq'),
    'time_psd_no_se':('time','freq'),
}
BRANCH_INPUTS={'time':'raw_eeg_microvolts','freq':'welch_absolute_relative_psd','bio':'spectral_biomarkers'}
VARIANT_INPUT_NAMES={k:tuple(BRANCH_INPUTS[x] for x in v) for k,v in VARIANT_BRANCHES.items()}
# Full recommended evaluation. If running a subset, the notebook will clearly show it as partial.
SELECTED_VARIANTS=list(VARIANT_BRANCHES)
ABLATION_ROOT=WORK_ROOT/'reviewer_notebook2_subjectwise_ablation'
ABLATION_ROOT.mkdir(parents=True,exist_ok=True)
INCLUDE_MODEL_WEIGHTS_IN_ZIP=True

In [ ]:
def build_variant_model(variant):
    if variant == 'full_original':
        return build_model()
    if variant not in VARIANT_BRANCHES:
        raise ValueError('Unknown variant: ' + str(variant))
    selected = VARIANT_BRANCHES[variant]
    use_se = variant not in {'time_only_no_se','time_psd_no_se','full_no_se'}
    raw_input = layers.Input((1280, 19, 1), name="raw_eeg_microvolts")
    welch_input = layers.Input((89, 19, 2), name="welch_absolute_relative_psd")
    biomarker_input = layers.Input((10,), name="spectral_biomarkers")

    # Temporal branch
    t0 = ChannelSegmentZScoreClip(name="temporal_zscore")(raw_input)
    t1 = cgn(t0, 32, (32, 1), 8, "time32")
    t2 = cgn(t0, 32, (64, 1), 8, "time64")
    t3 = cgn(t0, 32, (128, 1), 8, "time128")
    t = layers.Concatenate(name="time_concat")([t1, t2, t3])
    t = layers.DepthwiseConv2D((1, 19), depth_multiplier=2, use_bias=False, name="time_spatial")(t)
    t = layers.GroupNormalization(groups=16, name="time_spatial_gn")(t)
    t = layers.Activation("elu")(t)
    if use_se: t = se_block(t, 8, "time_se1")
    t = layers.AveragePooling2D((4, 1))(t)
    t = layers.SpatialDropout2D(0.15)(t)
    t = cgn(t, 192, (16, 1), 16, "time_sep1", separable=True)
    t = layers.AveragePooling2D((4, 1))(t)
    t = layers.SpatialDropout2D(0.20)(t)
    t = cgn(t, 256, (8, 1), 16, "time_sep2", separable=True)
    if use_se: t = se_block(t, 8, "time_se2")
    t = layers.AveragePooling2D((2, 1))(t)
    t = layers.SpatialDropout2D(0.25)(t)
    t = layers.GlobalAveragePooling2D(name="time_embedding")(t)

    # Welch branch
    f = cgn(welch_input, 32, (7, 1), 8, "freq1")
    f = cgn(f, 48, (5, 1), 8, "freq2")
    f = layers.DepthwiseConv2D((1, 19), depth_multiplier=2, use_bias=False, name="freq_spatial")(f)
    f = layers.GroupNormalization(groups=16, name="freq_spatial_gn")(f)
    f = layers.Activation("elu")(f)
    if use_se: f = se_block(f, 8, "freq_se1")
    f = layers.AveragePooling2D((2, 1))(f)
    f = layers.SpatialDropout2D(0.15)(f)
    f = cgn(f, 128, (5, 1), 16, "freq_sep1", separable=True)
    f = layers.AveragePooling2D((2, 1))(f)
    f = layers.SpatialDropout2D(0.20)(f)
    f = cgn(f, 192, (3, 1), 16, "freq_sep2", separable=True)
    if use_se: f = se_block(f, 8, "freq_se2")
    f = layers.SpatialDropout2D(0.25)(f)
    f = layers.GlobalAveragePooling2D(name="freq_embedding")(f)

    # Biomarkers
    b = layers.Dense(32, activation="gelu", name="bio_dense1")(biomarker_input)
    b = layers.Dropout(0.15)(b)
    b = layers.Dense(32, activation="gelu", name="bio_embedding")(b)

    # Fusion
    branch_values = {'time':t, 'freq':f, 'bio':b}
    included = [branch_values[k] for k in selected]
    z = included[0] if len(included)==1 else layers.Concatenate(name="fusion")(included)
    z = layers.Dense(256, activation="gelu", name="fusion_dense1")(z)
    z = layers.LayerNormalization(name="fusion_ln1")(z)
    z = layers.Dropout(0.30)(z)
    z = layers.Dense(96, activation="gelu", name="fusion_dense2")(z)
    z = layers.LayerNormalization(name="fusion_ln2")(z)
    z = layers.Dropout(0.20)(z)
    output = layers.Dense(3, activation="softmax", name="multiclass_output")(z)

    return Model(
        inputs={k:v for k,v in {
            'raw_eeg_microvolts':raw_input,
            'welch_absolute_relative_psd':welch_input,
            'spectral_biomarkers':biomarker_input,
        }.items() if k in VARIANT_INPUT_NAMES[variant]},
        outputs=output,
    )

In [ ]:
# Check that every model uses EXACTLY the intended inputs; original has no post-fusion SE.
class VariantSequence(MultiInputSequence):
    def __init__(self, frame, stats_dict, variant, **kwargs):
        self.variant=variant
        super().__init__(frame,stats_dict,**kwargs)
    def __getitem__(self,i):
        x,y=super().__getitem__(i)
        return {k:x[k] for k in VARIANT_INPUT_NAMES[self.variant]}, y

variant_models=[]
for variant in SELECTED_VARIANTS:
    tf.keras.backend.clear_session()
    model=build_variant_model(variant)
    inputs=sorted(x.name.split(':')[0].split('/')[-1] for x in model.inputs)
    assert inputs==sorted(VARIANT_INPUT_NAMES[variant]),(variant,inputs)
    assert model.output_shape[-1]==3
    variant_models.append({'variant':variant,'params':int(model.count_params()),
                           'se_in_branches':variant not in ['full_no_se','time_only_no_se','time_psd_no_se'],
                           'branches':'+'.join(VARIANT_BRANCHES[variant])})
    print(variant,model.count_params(),'parameters',inputs)
pd.DataFrame(variant_models).to_csv(ABLATION_ROOT/'variant_architectures.csv',index=False)
del model;tf.keras.backend.clear_session()

## Training all variant × subject-fold combinations

**No test leakage:** model selection uses only a held-out validation-subject partition within the outer development fold; test-subject labels are used only for evaluation. Outputs are saved immediately after each combination. In interrupted sessions, rerun with same dataset and configuration to reuse matching checkpoints (Colab storage must persist).

In [ ]:
# This cell DOES THE TRAINING. Run on Colab GPU. It may use substantial compute.
train_manifest=[]
for variant in SELECTED_VARIANTS:
    for fold,sets in fold_splits:
        target=ABLATION_ROOT/variant/f'fold_{fold:02d}'
        target.mkdir(parents=True,exist_ok=True)
        marker=target/'COMPLETE.json'
        signature={'variant':variant,'fold':fold,
            'sets':{k:sorted(v) for k,v in sets.items()},'meta_hash':metadata_sha256,
            'participants_hash':part_hash,'epochs':MAX_EPOCHS,'batch':BATCH_SIZE,
            'patience':PATIENCE,'lr':PEAK_LEARNING_RATE,'wd':WEIGHT_DECAY,
            'seed':MODEL_SEED,'label_smoothing':LABEL_SMOOTHING}
        fingerprint=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()
        required=['segment_predictions.csv','subject_predictions.csv','test_metrics.json','best.weights.h5']
        if marker.exists() and all((target/p).exists() for p in required):
            if json.loads(marker.read_text()).get('fingerprint')==fingerprint:
                print('RESUME',variant,'fold',fold)
                train_manifest.append(json.loads((target/'test_metrics.json').read_text()))
                continue
        marker.unlink(missing_ok=True)
        print('TRAIN',variant,'outer subject fold',fold)
        tr=frame_for_subjects(sets['train']);va=frame_for_subjects(sets['validation']);te=frame_for_subjects(sets['test'])
        assert set(tr.participant_id).isdisjoint(set(va.participant_id))
        assert set(tr.participant_id).isdisjoint(set(te.participant_id))
        assert set(va.participant_id).isdisjoint(set(te.participant_id))
        assert set(te.label)=={0,1,2}
        tf.keras.backend.clear_session();gc.collect()
        tf.keras.utils.set_random_seed(MODEL_SEED+fold)  # paired architecture starts
        norm=fold_normalizers(tr)
        train_seq=VariantSequence(tr,norm,variant,batch_size=BATCH_SIZE,shuffle=True,seed=MODEL_SEED+fold)
        valid_seq=VariantSequence(va,norm,variant,batch_size=BATCH_SIZE,shuffle=False)
        test_seq=VariantSequence(te,norm,variant,batch_size=BATCH_SIZE,shuffle=False)
        net=build_variant_model(variant)
        nsteps=int(math.ceil(len(tr)/BATCH_SIZE))
        net.compile(optimizer=tf.keras.optimizers.AdamW(
            learning_rate=WarmupCosineDecay(PEAK_LEARNING_RATE,MIN_LEARNING_RATE,WARMUP_EPOCHS*nsteps,MAX_EPOCHS*nsteps),
            weight_decay=WEIGHT_DECAY,clipnorm=GRADIENT_CLIP_NORM),
            loss=SparseLabelSmoothedCrossentropy(num_classes=3,label_smoothing=LABEL_SMOOTHING),
            metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
        callbacks=[TerminateOnNaN(),ValidationMacroF1(valid_seq),
            ModelCheckpoint(str(target/'best.weights.h5'),save_weights_only=True,save_best_only=True,
                            monitor='val_macro_f1',mode='max',verbose=0),
            EarlyStopping(monitor='val_macro_f1',mode='max',patience=PATIENCE,
                          min_delta=.001,verbose=1),CSVLogger(str(target/'training_history.csv'))]
        hist=net.fit(train_seq,validation_data=valid_seq,epochs=MAX_EPOCHS,verbose=2,callbacks=callbacks)
        net.load_weights(str(target/'best.weights.h5'))
        p=net.predict(test_seq,verbose=0)
        assert p.shape==(len(te),3) and np.isfinite(p).all() and np.allclose(p.sum(axis=1),1,atol=1e-4)
        pred=te[['segment_uid','participant_id','group','label','local_segment_index']].copy()
        pred.insert(0,'variant',variant);pred.insert(1,'fold',fold)
        for j,c in enumerate(CLASS_NAMES): pred['p_'+c]=p[:,j]
        pred['segment_prediction']=p.argmax(axis=1)
        pred['segment_correct']=(pred.label==pred.segment_prediction).astype(int)
        pred.to_csv(target/'segment_predictions.csv',index=False)
        sub=aggregate_subject_predictions(pred)
        sub.insert(0,'variant',variant)
        sub.to_csv(target/'subject_predictions.csv',index=False)
        seg_m=multiclass_metrics(pred.label,p)
        sub_m=multiclass_metrics(sub.label,sub[['p_HC','p_AD','p_FTD']].to_numpy())
        row={'variant':variant,'fold':fold,'n_test_subjects':len(sub),'n_test_segments':len(pred),
             'n_params':int(net.count_params()),'n_train_subjects':tr.participant_id.nunique(),
             'n_valid_subjects':va.participant_id.nunique(),
             'best_epoch':int(np.argmax(hist.history['val_macro_f1'])+1),
             **{'segment_'+k:float(v) for k,v in seg_m.items()},
             **{'subject_'+k:float(v) for k,v in sub_m.items()}}
        (target/'test_metrics.json').write_text(json.dumps(row,indent=2))
        np.savez_compressed(target/'training_normalization.npz',**norm)
        marker.write_text(json.dumps({'fingerprint':fingerprint,'variant':variant,'fold':fold},indent=2))
        train_manifest.append(row)
        print('SAVED',variant,fold,'subject accuracy',round(row['subject_accuracy'],4))
        del net,train_seq,valid_seq,test_seq,p,norm
        tf.keras.backend.clear_session();gc.collect()
print('Completed valid combinations:',len(train_manifest))

## Analysis — paired subject-level comparisons, Holm correction and uncertainty

Requires all selected variants to have completed **each of the five outer folds**. All variants must agree on the same 88 unique test subjects and labels. Pooled subject-level paired McNemar exact tests and class-stratified subject bootstrap CIs are primary; paired fold Wilcoxon tests are supplemental and have low power with 5 folds. Do not interpret segment-wise significance as independent-subject evidence.

In [ ]:
from scipy.stats import binomtest, wilcoxon
from scipy.stats import t as student_t
if not {'full_original','time_only','time_psd_no_se'}.issubset(SELECTED_VARIANTS):
    print('WARNING: mandatory review contrasts unavailable: include full_original, time_only, time_psd_no_se.')
rows=[]; subparts=[]; segparts=[]
for variant in SELECTED_VARIANTS:
    for fold,_ in fold_splits:
        target=ABLATION_ROOT/variant/f'fold_{fold:02d}'
        assert (target/'COMPLETE.json').exists(),f'Not yet trained: {variant}, fold {fold}'
        rows.append(json.loads((target/'test_metrics.json').read_text()))
        subparts.append(pd.read_csv(target/'subject_predictions.csv'))
        segparts.append(pd.read_csv(target/'segment_predictions.csv'))
all_fold=pd.DataFrame(rows); all_sub=pd.concat(subparts,ignore_index=True);all_seg=pd.concat(segparts,ignore_index=True)
for name,x in all_sub.groupby('variant'):
    assert x.participant_id.nunique()==88 and (x.groupby('participant_id').size()==1).all(),name
    assert x.groupby('group').size().to_dict()==EXPECTED_SUBJECT_COUNTS
reference=all_sub[all_sub.variant==SELECTED_VARIANTS[0]][['participant_id','label','fold']].sort_values('participant_id').reset_index(drop=True)
for name,x in all_sub.groupby('variant'):
    audit=x[['participant_id','label','fold']].sort_values('participant_id').reset_index(drop=True)
    assert reference.equals(audit),f'Fold/subject pairing mismatch for {name}'
all_fold.to_csv(ABLATION_ROOT/'all_variant_fold_metrics.csv',index=False)
all_sub.to_csv(ABLATION_ROOT/'all_variant_subject_oof_predictions.csv',index=False)
all_seg.to_csv(ABLATION_ROOT/'all_variant_segment_oof_predictions.csv',index=False)
summary=[]
for variant,x in all_fold.groupby('variant',sort=False):
    for metric in ['subject_accuracy','subject_macro_f1','subject_balanced_accuracy',
                   'subject_macro_auc_ovr','segment_accuracy','segment_macro_f1']:
        v=x[metric].to_numpy(float)
        sd=np.std(v,ddof=1); delta=student_t.ppf(.975,len(v)-1)*sd/np.sqrt(len(v))
        summary.append({'variant':variant,'metric':metric,'fold_mean':v.mean(),'fold_SD':sd,
                        'fold_CI95_descriptive_low':v.mean()-delta,'fold_CI95_descriptive_high':v.mean()+delta})
summary_df=pd.DataFrame(summary)
summary_df.to_csv(ABLATION_ROOT/'variant_fold_mean_sd_CI.csv',index=False)

planned=[('full_original','time_only'),('full_original','time_psd_no_se'),
         ('full_original','full_no_se'),('full_original','time_psd_se'),
         ('time_only','time_only_no_se'),('time_psd_se','time_psd_no_se'),
         ('full_original','biomarker_only'),('full_original','psd_only')]
contrasts=[(a,b) for a,b in planned if a in SELECTED_VARIANTS and b in SELECTED_VARIANTS]
rng=np.random.default_rng(8019)
comp=[]
for a,b in contrasts:
    xa=all_sub[all_sub.variant==a].set_index('participant_id').sort_index()
    xb=all_sub[all_sub.variant==b].set_index('participant_id').sort_index()
    assert np.array_equal(xa.label.values,xb.label.values)
    a_ok=xa.subject_correct.to_numpy(int);b_ok=xb.subject_correct.to_numpy(int)
    ab=int(((a_ok==1)&(b_ok==0)).sum());ba=int(((a_ok==0)&(b_ok==1)).sum())
    p_exact=binomtest(min(ab,ba),n=ab+ba,p=.5).pvalue if ab+ba else 1.
    fa=all_fold[all_fold.variant==a].sort_values('fold').subject_macro_f1.to_numpy(float)
    fb=all_fold[all_fold.variant==b].sort_values('fold').subject_macro_f1.to_numpy(float)
    try: p_wilcoxon=float(wilcoxon(fa-fb).pvalue) if np.any(fa!=fb) else 1.
    except ValueError:p_wilcoxon=np.nan
    label=xa.label.to_numpy(int)
    a_p=xa[['p_HC','p_AD','p_FTD']].to_numpy(float)
    b_p=xb[['p_HC','p_AD','p_FTD']].to_numpy(float)
    group_idx=[np.flatnonzero(label==k) for k in range(3)]
    acc_diffs=[];f1_diffs=[]
    for boot in range(1000):
        idx=np.concatenate([rng.choice(ids,size=len(ids),replace=True) for ids in group_idx])
        acc_diffs.append(float((a_p.argmax(1)[idx]==label[idx]).mean()-(b_p.argmax(1)[idx]==label[idx]).mean()))
        f1_diffs.append(float(f1_score(label[idx],a_p[idx].argmax(1),labels=[0,1,2],average='macro',zero_division=0)-
                              f1_score(label[idx],b_p[idx].argmax(1),labels=[0,1,2],average='macro',zero_division=0)))
    comp.append({'model_A':a,'model_B':b,
       'delta_subject_accuracy_A_minus_B':float(a_ok.mean()-b_ok.mean()),
       'delta_subject_accuracy_CI_low':float(np.quantile(acc_diffs,.025)),
       'delta_subject_accuracy_CI_high':float(np.quantile(acc_diffs,.975)),
       'delta_subject_macro_F1':float(f1_score(label,a_p.argmax(1),labels=[0,1,2],average='macro')-
                                     f1_score(label,b_p.argmax(1),labels=[0,1,2],average='macro')),
       'delta_subject_macro_F1_CI_low':float(np.quantile(f1_diffs,.025)),
       'delta_subject_macro_F1_CI_high':float(np.quantile(f1_diffs,.975)),
       'discordant_A_right_B_wrong':ab,'discordant_A_wrong_B_right':ba,
       'mcnemar_exact_p_uncorrected':p_exact,'paired_5fold_wilcoxon_F1_p_exploratory':p_wilcoxon})
compare=pd.DataFrame(comp)
if len(compare):
    # Holm step-down family-wise correction on exact subject-level paired tests
    order=np.argsort(compare.mcnemar_exact_p_uncorrected.to_numpy()); n=len(compare)
    adjusted=np.empty(n);cum=0.
    for rank,j in enumerate(order):
        cum=max(cum,min(1.,(n-rank)*float(compare.loc[j,'mcnemar_exact_p_uncorrected'])))
        adjusted[j]=cum
    compare['mcnemar_Holm_adjusted_p']=adjusted
    compare.to_csv(ABLATION_ROOT/'paired_model_comparisons_subject_level.csv',index=False)
    display(compare)
else: print('No registered contrasts selected.')
print('Summary:');display(summary_df[summary_df.metric=='subject_macro_f1'])

In [ ]:
# Publication figures and structured audit manifest
fig,ax=plt.subplots(figsize=(10,5))
plot=summary_df[summary_df.metric=='subject_macro_f1'].set_index('variant').loc[SELECTED_VARIANTS]
ax.bar(np.arange(len(plot)),plot.fold_mean,yerr=plot.fold_SD,capsize=4)
ax.set_xticks(np.arange(len(plot)),plot.index,rotation=35,ha='right')
ax.set_ylim(0,1.05);ax.set_ylabel('Mean subject-held-out macro F1 (± fold SD)')
ax.set_title('5-fold subject-disjoint three-class ablation')
fig.tight_layout();fig.savefig(ABLATION_ROOT/'subjectwise_ablation_macroF1.png',dpi=200);plt.show()
manifest={'notebook':'Notebook 2','dataset':'OpenNeuro ds004504','participants_sha256':part_hash,
 'clean_segment_metadata_sha256':metadata_sha256,'original_full_postfusion_se':False,
 'variants_planned':list(VARIANT_BRANCHES),'variants_executed':SELECTED_VARIANTS,
 'n_outer_folds':N_FOLDS,'outer_seed':OUTER_SEED,'inner_seed':INNER_SEED,
 'test_subjects_per_variant':88,'subject_overlap':0,
 'training_normalization':'recomputed on training subjects within each fold',
 'checkpoint_metric':'validation macro F1 on held-out validation subjects',
 'paired_tests':'subject McNemar exact + class-stratified bootstrap; Holm correction across listed contrasts',
 'limitations':['5 folds limits paired Wilcoxon power','subject-wise CV on one public cohort; not external validation',
                'comparisons differ from manuscript binary ablation design']}
(ABLATION_ROOT/'experiment_manifest.json').write_text(json.dumps(manifest,indent=2))
archive=WORK_ROOT/'Notebook2_ThreeClass_Subjectwise_Ablation_Results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for p in ABLATION_ROOT.rglob('*'):
        if p.is_file() and (INCLUDE_MODEL_WEIGHTS_IN_ZIP or not p.name.endswith('.weights.h5')):
            z.write(p,arcname=str(p.relative_to(ABLATION_ROOT)))
print('Saved:',archive,'size MB:',round(archive.stat().st_size/1e6,2))
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:pass

### Reviewer interpretation

The **full original architecture** should be supported only if subject-held-out performance and paired differences justify it. Report model parameter counts, paired subject-level differences, CIs, exact tests and Holm-adjusted p values. If full fusion is not better than temporal-only, acknowledge it. Do not add an unimplemented post-fusion SE block to the description. Actual results appear only after execution.